# Data Preprocessing

Prepare a clean dataset for recommendation-system development.

This notebook:

- reads the raw dataset without modifying it
- parses and deduplicates interests per user
- keeps `city` and `country` as separate location fields
- adds `city_country_key` for city disambiguation
- creates a reproducible `80/10/10` split
- writes `data/processed_data.csv`


In [1]:
from pathlib import Path
import ast
import json

import pandas as pd

RAW_DATA_CANDIDATES = [
    Path("../data/SocialMediaUsersDataset.csv"),  # when notebook is run from EDA/
    Path("data/SocialMediaUsersDataset.csv"),     # when notebook is run from repo root
]

RAW_DATA_PATH = next((path for path in RAW_DATA_CANDIDATES if path.exists()), None)
if RAW_DATA_PATH is None:
    raise FileNotFoundError("Could not find SocialMediaUsersDataset.csv in data/ or ../data/")

PROCESSED_DATA_PATH = RAW_DATA_PATH.parent / "processed_data.csv"
REFERENCE_DATE = pd.Timestamp("2026-07-27")
RANDOM_SEED = 42

RAW_DATA_PATH, PROCESSED_DATA_PATH


(PosixPath('../data/SocialMediaUsersDataset.csv'),
 PosixPath('../data/processed_data.csv'))

## 1. Load raw data

Read the raw file into memory. The raw CSV is not edited.

In [2]:
raw_df = pd.read_csv(RAW_DATA_PATH, dtype={"UserID": "string"})

print(f"Rows: {len(raw_df):,}")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()


Rows: 100,000
Columns: ['UserID', 'Name', 'Gender', 'DOB', 'Interests', 'City', 'Country']


,UserID,Name,Gender,DOB,Interests,City,Country
0,00001,Jesse Lawhorn,Female,1958-10-15,"'Movies', 'Fashion', 'Fashion', 'Books'",Sibolga,Indonesia
1,00002,Stacy Payne,Female,2004-07-21,"'Gaming', 'Finance and investments', 'Outdoor ...",Al Abyār,Libya
2,00003,Katrina Nicewander,Female,2000-02-07,"'DIY and crafts', 'Music', 'Science', 'Fashion'",Wādī as Sīr,Jordan
3,00004,Eric Yarbrough,Male,1985-04-14,"'Outdoor activities', 'Cars and automobiles'",Matera,Italy
4,00005,Daniel Adkins,Female,1955-09-18,"'Politics', 'History'",Biruaca,Venezuela


## 2. Parse and deduplicate interests

Each user's interest list is parsed from the raw string. Repeated interests for the same user are kept only once.

In [3]:
def parse_interests(raw: str) -> list[str]:
    """
    Convert the raw `Interests` field into a clean list of interest labels.

    The raw dataset stores interests as one quoted, comma-separated string,
    for example: "'Movies', 'Fashion', 'Fashion', 'Books'".

    This function parses that string into a Python list, strips surrounding
    whitespace, removes empty values, and returns an empty list if the raw
    value is missing.
    """
    if pd.isna(raw):
        return []

    try:
        values = ast.literal_eval("[" + str(raw) + "]")
    except Exception:
        values = [part.strip().strip("\"'") for part in str(raw).split(",")]

    return [str(value).strip() for value in values if str(value).strip()]


def dedupe_preserve_order(values: list[str]) -> list[str]:
    """
    Remove duplicate interests for a single user while preserving original order.

    Repeated interests in one user's profile are treated as duplicate data, not
    stronger preference signal. The first occurrence is kept and later repeats
    are removed.

    Example:
        ["Movies", "Fashion", "Fashion", "Books"]
        -> ["Movies", "Fashion", "Books"]
    """
    seen = set()
    deduped = []

    for value in values:
        if value not in seen:
            seen.add(value)
            deduped.append(value)

    return deduped


parsed_interests = raw_df["Interests"].apply(parse_interests)
deduped_interests = parsed_interests.apply(dedupe_preserve_order)

interest_preprocessing_report = pd.DataFrame(
    [
        {"metric": "rows", "value": len(raw_df)},
        {"metric": "rows_with_zero_parsed_interests", "value": int((parsed_interests.apply(len) == 0).sum())},
        {"metric": "rows_with_duplicate_interests_removed", "value": int((parsed_interests.apply(len) != deduped_interests.apply(len)).sum())},
        {"metric": "unique_interests_after_deduplication", "value": len(set(interest for values in deduped_interests for interest in values))},
    ]
)

interest_preprocessing_report


,metric,value
0,rows,100000
1,rows_with_zero_parsed_interests,0
2,rows_with_duplicate_interests_removed,12897
3,unique_interests_after_deduplication,29


## 3. Build processed dataset

Create model-friendly columns while keeping the original raw data separate.

In [4]:
dob = pd.to_datetime(raw_df["DOB"], errors="coerce")
age = ((REFERENCE_DATE - dob).dt.days / 365.25).astype("int64")

processed_df = pd.DataFrame(
    {
        "user_id": raw_df["UserID"].astype("string"),
        "name": raw_df["Name"].astype("string").str.strip(),
        "gender": raw_df["Gender"].astype("string").str.strip(),
        "dob": raw_df["DOB"].astype("string"),
        "age": age,
        "city": raw_df["City"].astype("string").str.strip(),
        "country": raw_df["Country"].astype("string").str.strip(),
        "interests_raw": raw_df["Interests"].astype("string"),
        "interests_list": parsed_interests.apply(json.dumps),
        "interests_deduped": deduped_interests.apply(json.dumps),
        "interest_count": deduped_interests.apply(len),
    }
)

processed_df["city_country_key"] = processed_df["city"] + ", " + processed_df["country"]

processed_df = processed_df[
    [
        "user_id",
        "name",
        "gender",
        "dob",
        "age",
        "city",
        "country",
        "city_country_key",
        "interests_raw",
        "interests_list",
        "interests_deduped",
        "interest_count",
    ]
]

processed_df.head()


,user_id,name,gender,dob,age,city,country,city_country_key,interests_raw,interests_list,interests_deduped,interest_count
0,00001,Jesse Lawhorn,Female,1958-10-15,67,Sibolga,Indonesia,"Sibolga, Indonesia","'Movies', 'Fashion', 'Fashion', 'Books'","[""Movies"", ""Fashion"", ""Fashion"", ""Books""]","[""Movies"", ""Fashion"", ""Books""]",3
1,00002,Stacy Payne,Female,2004-07-21,22,Al Abyār,Libya,"Al Abyār, Libya","'Gaming', 'Finance and investments', 'Outdoor ...","[""Gaming"", ""Finance and investments"", ""Outdoor...","[""Gaming"", ""Finance and investments"", ""Outdoor...",4
2,00003,Katrina Nicewander,Female,2000-02-07,26,Wādī as Sīr,Jordan,"Wādī as Sīr, Jordan","'DIY and crafts', 'Music', 'Science', 'Fashion'","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]",4
3,00004,Eric Yarbrough,Male,1985-04-14,41,Matera,Italy,"Matera, Italy","'Outdoor activities', 'Cars and automobiles'","[""Outdoor activities"", ""Cars and automobiles""]","[""Outdoor activities"", ""Cars and automobiles""]",2
4,00005,Daniel Adkins,Female,1955-09-18,70,Biruaca,Venezuela,"Biruaca, Venezuela","'Politics', 'History'","[""Politics"", ""History""]","[""Politics"", ""History""]",2


## 4. Add train/validation/test split

Create a reproducible user-level `80/10/10` split.

In [5]:
shuffled_indices = processed_df.sample(frac=1, random_state=RANDOM_SEED).index

n_total = len(processed_df)
n_train = int(n_total * 0.80)
n_validation = int(n_total * 0.10)

train_indices = shuffled_indices[:n_train]
validation_indices = shuffled_indices[n_train : n_train + n_validation]
test_indices = shuffled_indices[n_train + n_validation :]

processed_df["split"] = "test"
processed_df.loc[train_indices, "split"] = "train"
processed_df.loc[validation_indices, "split"] = "validation"

split_report = (
    processed_df["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="row_count")
)
split_report["percentage"] = (split_report["row_count"] / len(processed_df) * 100).round(2)

split_report


,split,row_count,percentage
0,train,80000,80.0
1,validation,10000,10.0
2,test,10000,10.0


## 5. Save processed data

Write the derived dataset to `data/processed_data.csv`.

In [6]:
processed_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Saved processed data to: {PROCESSED_DATA_PATH}")
print(f"Rows written: {len(processed_df):,}")


Saved processed data to: ../data/processed_data.csv
Rows written: 100,000


## 6. Validate processed output

Confirm that the processed file was created and matches the expected row count, columns, and split distribution.

In [7]:
saved_df = pd.read_csv(PROCESSED_DATA_PATH, dtype={"user_id": "string"})

expected_columns = [
    "user_id",
    "name",
    "gender",
    "dob",
    "age",
    "city",
    "country",
    "city_country_key",
    "interests_raw",
    "interests_list",
    "interests_deduped",
    "interest_count",
    "split",
]

validation_report = pd.DataFrame(
    [
        {"check": "processed file exists", "passed": PROCESSED_DATA_PATH.exists(), "details": str(PROCESSED_DATA_PATH)},
        {"check": "row count matches raw data", "passed": len(saved_df) == len(raw_df), "details": len(saved_df)},
        {"check": "expected columns present", "passed": list(saved_df.columns) == expected_columns, "details": list(saved_df.columns)},
        {"check": "user_id remains unique", "passed": saved_df["user_id"].is_unique, "details": int(saved_df["user_id"].duplicated().sum())},
        {"check": "no missing core fields", "passed": saved_df[["user_id", "city", "country", "interests_deduped", "split"]].isna().sum().sum() == 0, "details": saved_df[["user_id", "city", "country", "interests_deduped", "split"]].isna().sum().to_dict()},
        {"check": "split values are valid", "passed": set(saved_df["split"].unique()) == {"train", "validation", "test"}, "details": sorted(saved_df["split"].unique())},
    ]
)

validation_report


,check,passed,details
0,processed file exists,True,../data/processed_data.csv
1,row count matches raw data,True,100000
2,expected columns present,True,"[user_id, name, gender, dob, age, city, countr..."
3,user_id remains unique,True,0
4,no missing core fields,True,"{'user_id': 0, 'city': 0, 'country': 0, 'inter..."
5,split values are valid,True,"[test, train, validation]"


## Preprocessing summary

- Raw data remains unchanged.
- Processed data is saved to `data/processed_data.csv`.
- `city` and `country` are preserved separately.
- `city_country_key` is added for disambiguation.
- Repeated interests are removed per user.
- `split` provides a reproducible `80/10/10` user-level split.
